In [12]:
from bs4 import BeautifulSoup
import requests
import re
import pandas as pd

# URL for IMDb top 250 movies
url = 'http://www.imdb.com/chart/top'

# Set up headers with a user-agent
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/85.0.4183.121 Safari/537.36'
}

# Send the request with headers
response = requests.get(url, headers=headers)

# Check if the request was successful
if response.status_code != 200:
    print("Failed to retrieve the webpage. Status code:", response.status_code)
else:
    # Print the raw HTML for debugging
    print(response.text[:1000])  # Print the first 1000 characters of the HTML

    soup = BeautifulSoup(response.text, "html.parser")

    # Selectors for extracting data
    movies = soup.select('td.titleColumn')
    crew = [a.attrs.get('title') for a in soup.select('td.titleColumn a')]
    ratings = [b.attrs.get('data-value') for b in soup.select('td.posterColumn span[name=ir]')]

    # Create an empty list for storing movie information
    movies_list = []

    # Check if we have extracted any data
    if not movies or not crew or not ratings:
        print("No movie data found.")
    else:
        # Iterating over movies to extract each movie's details
        for index in range(len(movies)):
            # Separating movie into: 'place', 'title', 'year'
            movie_string = movies[index].get_text()
            movie = (' '.join(movie_string.split()).replace('.', ''))
            
            movie_title = movie[len(str(index)) + 1:-7]
            year = re.search(r'\((.*?)\)', movie_string).group(1)
            place = movie[:len(str(index))]

            # Creating a dictionary for the movie data
            data = {
                "place": place,
                "movie_title": movie_title,
                "rating": ratings[index],
                "year": year,
                "star_cast": crew[index],
            }
            movies_list.append(data)

    # Check if any movies were added to the list
    if not movies_list:
        print("No movie details found.")
    else:
        # Printing movie details with its rating
        for movie in movies_list:
            print(movie['place'], '-', movie['movie_title'], '(' + movie['year'] + ') -', 'Starring:', movie['star_cast'], movie['rating'])

        # Save the data to a CSV file
        df = pd.DataFrame(movies_list)
        df.to_csv('imdb_top_250_movies.csv', index=False)
        print("Data saved to 'imdb_top_250_movies.csv'.")


<!DOCTYPE html><html lang="en-US" xmlns:og="http://opengraphprotocol.org/schema/" xmlns:fb="http://www.facebook.com/2008/fbml"><head><meta charSet="utf-8"/><meta name="viewport" content="width=device-width"/><script>if(typeof uet === 'function'){ uet('bb', 'LoadTitle', {wb: 1}); }</script><script>window.addEventListener('load', (event) => {
        if (typeof window.csa !== 'undefined' && typeof window.csa === 'function') {
            var csaLatencyPlugin = window.csa('Content', {
                element: {
                    slotId: 'LoadTitle',
                    type: 'service-call'
                }
            });
            csaLatencyPlugin('mark', 'clickToBodyBegin', 1728403387151);
        }
    })</script><title>IMDb Top 250 Movies</title><meta name="description" content="As rated by regular IMDb voters." data-id="main"/><script type="application/ld+json">{"@type":"ItemList","itemListElement":[{"@type":"ListItem","item":{"@type":"Movie","url":"https://www.imdb.com/title/tt